# Сравнение моделей

На предыдущем этапе были построены и проверены различные группы признаков. Наиболее устойчивый результат показали временные, событийные, технические и указательные. Дополнительные признаки, связанные с объявлениями и поиском, почти не изменили средний результат временной валидации, но улучшили качество на наиболее позднем временном отрезке.

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score

from metric import precision_at_recall

In [2]:
train = pd.read_csv("data/train.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
test = pd.read_csv("data/test.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

Для каждого объекта разрешено использовать только события, произошедшие внутри заданного суточного окна. Поэтому перед построением признаков отдельно сформируем события для train и test.

In [3]:
def events_in_window(events, meta):
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner")
    ev = ev[(ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])].copy()
    ev = ev.drop(columns=["window_start_ts", "window_end_ts"])
    return ev

In [4]:
ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)

print("Train events:", ev_train.shape)
print("Test events:", ev_test.shape)

Train events: (198436, 14)
Test events: (89690, 14)


Из метаданных используем возраст куки к началу окна наблюдения

In [5]:
def make_meta_features(meta):
    f = meta[["cookie_id"]].copy()
    f["cookie_age_hours"] = (meta["window_start_ts"] - meta["cookie_created_at"]).dt.total_seconds() / 3600
    f["log_cookie_age_hours"] = np.log1p(f["cookie_age_hours"].clip(lower=0))
    return f

Добавим общие характеристики активности: число событий и разнообразие объявлений, категорий, локаций и типов продавцов

In [6]:
def make_basic_features(ev, meta):
    g = ev.groupby("cookie_id")
    f = pd.DataFrame(index=g.size().index)
    f["n_events"] = g.size()
    f["event_type_nunique"] = g["event_name"].nunique()
    f["item_nunique"] = g["item_id"].nunique()
    f["category_nunique"] = g["item_category"].nunique()
    f["location_nunique"] = g["item_location"].nunique()
    f["seller_type_nunique"] = g["seller_type"].nunique()
    f = meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left")
    f = f.fillna(0)
    return f

Временные характеристики дали один из самых больших приростов качества. Боты чаще совершали действия с небольшими интервалами и выполняли больше событий за сопоставимый период активности

In [7]:
def make_time_features(ev, meta):
    x = ev[["cookie_id", "event_ts"]].copy()
    x = x.sort_values(["cookie_id", "event_ts"])
    x["gap_sec"] = x.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    x["gap_le_5"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 5, np.nan)
    x["gap_le_10"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 10, np.nan)
    x["gap_le_30"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 30, np.nan)
    g = x.groupby("cookie_id")
    f = g.agg(first_event_ts=("event_ts", "min"), last_event_ts=("event_ts", "max"), mean_gap_sec=("gap_sec", "mean"), median_gap_sec=("gap_sec", "median"), std_gap_sec=("gap_sec", "std"), min_gap_sec=("gap_sec", "min"), max_gap_sec=("gap_sec", "max"), share_gap_le_5=("gap_le_5", "mean"), share_gap_le_10=("gap_le_10", "mean"), share_gap_le_30=("gap_le_30", "mean")).reset_index()
    f["active_span_sec"] = (f["last_event_ts"] - f["first_event_ts"]).dt.total_seconds()
    f = f.drop(columns=["first_event_ts", "last_event_ts"])
    event_count = ev.groupby("cookie_id").size().rename("n_events_tmp").reset_index()
    f = f.merge(event_count, on="cookie_id", how="left")
    f["events_per_active_hour"] = np.where(f["active_span_sec"] > 0, (f["n_events_tmp"] - 1) / (f["active_span_sec"] / 3600), 0)
    f = f.drop(columns=["n_events_tmp"])
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

Одного общего числа событий недостаточно: люди и боты отличаются и структурой действий. Поэтому для каждого типа события используем абсолютное количество и его долю среди всех событий куки

In [9]:
def make_event_count_features(ev, meta):
    counts = ev.groupby(["cookie_id", "event_name"]).size().unstack(fill_value=0)
    counts = counts.add_prefix("count_")
    f = meta[["cookie_id"]].merge(counts.reset_index(), on="cookie_id", how="left")
    f = f.fillna(0)
    count_cols = [col for col in f.columns if col.startswith("count_")]
    f["event_count_total"] = f[count_cols].sum(axis=1)

    for col in count_cols:
        f["share_" + col[6:]] = np.where(f["event_count_total"] > 0, f[col] / f["event_count_total"], 0)

    f = f.drop(columns=["event_count_total"])
    return f

Для просмотров объявлений отдельно учитываем количество просмотров, число уникальных объявлений и повторные просмотры

In [8]:
def make_item_features(ev, meta):
    item_view = ev[ev["event_name"] == "item_view"].copy()
    f = item_view.groupby("cookie_id").agg(item_view_count=("item_id", "size"), item_view_nunique=("item_id", "nunique")).reset_index()
    f["item_view_repeat_count"] = f["item_view_count"] - f["item_view_nunique"]
    f["item_view_unique_ratio"] = np.where(f["item_view_count"] > 0, f["item_view_nunique"] / f["item_view_count"], 0)
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

В EDA боты чаще проходили несколько страниц одного поискового запроса и в среднем глубже просматривали выдачу. Поэтому отдельно добавим характеристики поисков и пагинации

In [10]:
def make_search_features(ev, meta):
    search = ev[ev["event_name"] == "search_results_view"].copy()
    f = search.groupby("cookie_id").agg(search_query_nunique=("search_query", "nunique"), search_page_mean=("search_page", "mean"), search_page_median=("search_page", "median"), search_page_max=("search_page", "max"), search_page_nunique=("search_page", "nunique")).reset_index()
    search_count = search.groupby("cookie_id").size().rename("search_event_count").reset_index()
    f = f.merge(search_count, on="cookie_id", how="outer")
    f["searches_per_query"] = np.where(f["search_query_nunique"] > 0, f["search_event_count"] / f["search_query_nunique"], 0)
    search["page_gt_1"] = search["search_page"] > 1
    search["page_gt_3"] = search["search_page"] > 3
    search["page_gt_5"] = search["search_page"] > 5
    depth = search.groupby("cookie_id").agg(share_page_gt_1=("page_gt_1", "mean"), share_page_gt_3=("page_gt_3", "mean"), share_page_gt_5=("page_gt_5", "mean")).reset_index()
    f = f.merge(depth, on="cookie_id", how="outer")
    query_page = search.dropna(subset=["search_query", "search_page"]).drop_duplicates(["cookie_id", "search_query", "search_page"])
    query_page_nunique = query_page.groupby("cookie_id").size().rename("query_page_nunique").reset_index()
    f = f.merge(query_page_nunique, on="cookie_id", how="outer")
    pages_per_query = query_page.groupby(["cookie_id", "search_query"]).size().rename("pages_for_query").reset_index()
    pages_summary = pages_per_query.groupby("cookie_id")["pages_for_query"].agg(["mean", "max"]).reset_index()
    pages_summary.columns = ["cookie_id", "pages_per_query_mean", "pages_per_query_max"]
    f = f.merge(pages_summary, on="cookie_id", how="outer")
    f["unique_query_page_ratio"] = np.where(f["search_event_count"] > 0, f["query_page_nunique"] / f["search_event_count"], 0)
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

Технические характеристики по отдельности оказались не очень сильными, но улучшали результат в сочетании с другими группами признаков. Используем платформу и несколько обобщённых характеристик User-Agent, не привязываясь к конкретным версиям браузеров

In [11]:
def make_tech_features(ev, meta):
    x = ev[["cookie_id", "platform", "user_agent"]].copy()
    x["platform_norm"] = x["platform"].astype("string").str.strip().str.lower()
    x["ua_lower"] = x["user_agent"].str.lower()
    x["is_headless"] = x["ua_lower"].str.contains("headlesschrome")
    x["is_firefox"] = x["ua_lower"].str.contains("firefox")
    x["is_yandex"] = x["ua_lower"].str.contains("yabrowser")
    x["is_avito_app"] = x["ua_lower"].str.startswith("avito/")
    x["is_chrome"] = x["ua_lower"].str.contains("chrome") & ~x["is_headless"] & ~x["is_yandex"]
    x["is_windows"] = x["ua_lower"].str.contains("windows")
    x["is_linux"] = x["ua_lower"].str.contains("linux") & ~x["ua_lower"].str.contains("android")
    x["is_macos"] = x["ua_lower"].str.contains("macintosh")
    x["is_android_ua"] = x["ua_lower"].str.contains("android")
    x["ua_length"] = x["user_agent"].str.len()
    f = x.groupby("cookie_id").agg(platform_nunique=("platform_norm", "nunique"), ua_nunique=("user_agent", "nunique"), ua_length_mean=("ua_length", "mean"), ua_length_min=("ua_length", "min"), ua_length_max=("ua_length", "max"), is_headless=("is_headless", "max"), is_firefox=("is_firefox", "max"), is_yandex=("is_yandex", "max"), is_avito_app=("is_avito_app", "max"), is_chrome=("is_chrome", "max"), is_windows=("is_windows", "max"), is_linux=("is_linux", "max"), is_macos=("is_macos", "max"), is_android_ua=("is_android_ua", "max")).reset_index()
    platform_counts = x.groupby(["cookie_id", "platform_norm"]).size().unstack(fill_value=0)
    platform_counts = platform_counts.div(platform_counts.sum(axis=1), axis=0)
    platform_counts = platform_counts.add_prefix("platform_share_").reset_index()
    f = f.merge(platform_counts, on="cookie_id", how="left")
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

Признаки указателя дали самый большой дополнительный прирост качества. У ботов координаты указателя встречались реже, занимали меньшую область экрана, а расстояния между последовательными позициями были заметно меньше

In [12]:
def make_pointer_features(ev, meta):
    x = ev[["cookie_id", "event_ts", "platform", "pointer_x", "pointer_y"]].copy()
    x["platform_norm"] = x["platform"].astype("string").str.strip().str.lower()
    x["has_pointer"] = x["pointer_x"].notna() & x["pointer_y"].notna()
    basic = x.groupby("cookie_id").agg(pointer_event_count=("has_pointer", "sum"), pointer_event_share=("has_pointer", "mean")).reset_index()
    basic["has_any_pointer"] = basic["pointer_event_count"] > 0
    valid = x[x["has_pointer"]].copy()
    coord = valid.groupby("cookie_id").agg(pointer_x_mean=("pointer_x", "mean"), pointer_x_std=("pointer_x", "std"), pointer_x_min=("pointer_x", "min"), pointer_x_max=("pointer_x", "max"), pointer_y_mean=("pointer_y", "mean"), pointer_y_std=("pointer_y", "std"), pointer_y_min=("pointer_y", "min"), pointer_y_max=("pointer_y", "max")).reset_index()
    coord["pointer_x_range"] = coord["pointer_x_max"] - coord["pointer_x_min"]
    coord["pointer_y_range"] = coord["pointer_y_max"] - coord["pointer_y_min"]
    valid["_row_order"] = np.arange(len(valid))
    valid = valid.sort_values(["cookie_id", "event_ts", "_row_order"])
    valid["prev_x"] = valid.groupby("cookie_id")["pointer_x"].shift()
    valid["prev_y"] = valid.groupby("cookie_id")["pointer_y"].shift()
    valid["dx"] = valid["pointer_x"] - valid["prev_x"]
    valid["dy"] = valid["pointer_y"] - valid["prev_y"]
    valid["move_distance"] = np.sqrt(valid["dx"] ** 2 + valid["dy"] ** 2)
    moves = valid.groupby("cookie_id").agg(move_distance_mean=("move_distance", "mean"), move_distance_median=("move_distance", "median"), move_distance_std=("move_distance", "std"), move_distance_max=("move_distance", "max")).reset_index()
    pairs = valid.drop_duplicates(["cookie_id", "pointer_x", "pointer_y"]).groupby("cookie_id").size().rename("pointer_pair_nunique").reset_index()
    counts = valid.groupby("cookie_id").size().rename("pointer_count").reset_index()
    repeat = counts.merge(pairs, on="cookie_id", how="left")
    repeat["pointer_unique_ratio"] = np.where(repeat["pointer_count"] > 0, repeat["pointer_pair_nunique"] / repeat["pointer_count"], 0)
    platform_pointer = x.groupby(["cookie_id", "platform_norm"])["has_pointer"].mean().unstack(fill_value=0)
    platform_pointer = platform_pointer.add_prefix("pointer_share_").reset_index()
    f = basic.merge(coord, on="cookie_id", how="left")
    f = f.merge(moves, on="cookie_id", how="left")
    f = f.merge(repeat[["cookie_id", "pointer_unique_ratio"]], on="cookie_id", how="left")
    f = f.merge(platform_pointer, on="cookie_id", how="left")
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

Теперь построим все выбранные группы признаков для train и test и объединим их по cookie_id

In [13]:
def merge_feature_groups(meta, feature_groups):
    result = meta[["cookie_id"]].copy()

    for features in feature_groups:
        result = result.merge(features, on="cookie_id", how="left")

    return result

In [14]:
basic_train_features = make_basic_features(ev_train, train)
basic_test_features = make_basic_features(ev_test, test)

meta_train_features = make_meta_features(train)
meta_test_features = make_meta_features(test)

time_train_features = make_time_features(ev_train, train)
time_test_features = make_time_features(ev_test, test)

event_train_features = make_event_count_features(ev_train, train)
event_test_features = make_event_count_features(ev_test, test)

tech_train_features = make_tech_features(ev_train, train)
tech_test_features = make_tech_features(ev_test, test)

pointer_train_features = make_pointer_features(ev_train, train)
pointer_test_features = make_pointer_features(ev_test, test)

item_train_features = make_item_features(ev_train, train)
item_test_features = make_item_features(ev_test, test)

search_train_features = make_search_features(ev_train, train)
search_test_features = make_search_features(ev_test, test)

In [16]:
train_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, tech_train_features, pointer_train_features, item_train_features, search_train_features])
test_features = merge_feature_groups(test, [basic_test_features, meta_test_features, time_test_features, event_test_features, tech_test_features, pointer_test_features, item_test_features, search_test_features])

# Сравнение моделей

Все модели будем оценивать на одной и той же временной схеме. Каждый следующий фолд расположен строго позже данных, использованных для обучения. Такая схема лучше соответствует реальной постановке, поскольку тестовая выборка также находится в будущем относительно трейна

In [17]:
temporal_folds = [("2026-04-11", "2026-04-13"), ("2026-04-14", "2026-04-16"), ("2026-04-17", "2026-04-19")]

Чтобы сравнение было честным, для всех моделей используем одну функцию. Основной метрикой остаётся максимальный Precision при Recall не ниже 70%. ROC-AUC и PR-AUC выводим как дополнительные диагностические показатели.

In [18]:
def evaluate_model_temporal(features, model_factory, name):
    cols = [col for col in features.columns if col != "cookie_id"]
    y = train["target"].values
    dates = train["window_start_ts"]
    fold_results = []

    for fold_id, (valid_start, valid_end) in enumerate(temporal_folds, start=1):
        train_mask = dates < valid_start
        valid_mask = dates.between(valid_start, valid_end)
        model = model_factory()
        model.fit(features.loc[train_mask, cols], y[train_mask])
        pred = model.predict_proba(features.loc[valid_mask, cols])[:, 1]
        score = precision_at_recall(y[valid_mask], pred)
        roc_auc = roc_auc_score(y[valid_mask], pred)
        pr_auc = average_precision_score(y[valid_mask], pred)
        fold_results.append({"name": name, "fold": fold_id, "precision_at_recall_70": score, "roc_auc": roc_auc, "pr_auc": pr_auc})

    return pd.DataFrame(fold_results)

## Extra Trees

Extra Trees похож на Random Forest, но добавляет больше случайности при выборе порогов разбиения

In [19]:
def make_extra_trees():
    return ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3, max_features="sqrt", random_state=0, n_jobs=-1)

In [20]:
extra_trees_cv = evaluate_model_temporal(train_features, make_extra_trees, "Extra Trees")
extra_trees_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,Extra Trees,1,0.573171,0.899161,0.701203
1,Extra Trees,2,0.595652,0.907808,0.710182
2,Extra Trees,3,0.700000,0.907074,0.716548


Для Extra Trees также проверим вариант без случайного ограничения количества признаков на split

In [21]:
def make_extra_trees_all_features():
    return ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3, max_features=1.0, random_state=0, n_jobs=-1)

In [22]:
extra_trees_all_cv = evaluate_model_temporal(train_features, make_extra_trees_all_features, "Extra Trees max_features=1.0")
extra_trees_all_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,Extra Trees max_features=1.0,1,0.528302,0.896218,0.717644
1,Extra Trees max_features=1.0,2,0.628440,0.907972,0.720154
2,Extra Trees max_features=1.0,3,0.724359,0.908553,0.733058


Extra Trees не улучшил качество относительно Random Forest. Использование всех признаков при выборе разбиения немного повысило средний результат Extra Trees, однако модель всё равно уступает Random Forest на временной валидации. Поэтому дальше сосредоточимся на моделях градиентного бустинга

# Градиентный бустинг

Random Forest показал хороший результат, однако деревья в нём обучаются независимо друг от друга. В градиентном бустинге каждое следующее дерево строится с учётом ошибок уже построенного ансамбля

In [23]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [24]:
def make_hist_gradient_boosting():
    return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, random_state=0)

Начнём без подбора гиперпараметров. Используем небольшой learning rate и ограничиваем сложность отдельных деревьев, чтобы сначала оценить сам потенциал градиентного бустинга на наших признаках.

In [25]:
hist_gb_cv = evaluate_model_temporal(train_features, make_hist_gradient_boosting, "HistGradientBoosting")
hist_gb_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,HistGradientBoosting,1,0.679612,0.909151,0.739668
1,HistGradientBoosting,2,0.676471,0.927799,0.755601
2,HistGradientBoosting,3,0.746667,0.924477,0.760427


In [26]:
def make_hist_gradient_boosting_deeper():
    return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_leaf_nodes=63, min_samples_leaf=10, l2_regularization=1.0, random_state=0)

Во втором варианте разрешим деревьям строить более детальные разбиения и уменьшим минимальное число объектов в листе. Это позволит проверить, не ограничивает ли предыдущая конфигурация сложные взаимодействия между признаками

In [27]:
hist_gb_deeper_cv = evaluate_model_temporal(train_features, make_hist_gradient_boosting_deeper, "HistGradientBoosting deeper")
hist_gb_deeper_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,HistGradientBoosting deeper,1,0.689655,0.899628,0.736004
1,HistGradientBoosting deeper,2,0.681592,0.921245,0.749429
2,HistGradientBoosting deeper,3,0.772414,0.926521,0.765444


Градиентный бустинг заметно улучшил результат относительно ансамблей независимых деревьев. Более глубокий вариант оказался лучшим на всех трёх временных фолдах, поэтому далее будем использовать его как новую контрольную модель и проверим более специализированные реализации градиентного бустинга

## XGBoost

In [30]:
from xgboost import XGBClassifier

In [31]:
def make_xgboost():
    return XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, min_child_weight=10, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.0, eval_metric="logloss", random_state=0, n_jobs=-1)

Начнём с умеренно регуляризованной модели

In [32]:
xgboost_cv = evaluate_model_temporal(train_features, make_xgboost, "XGBoost")
xgboost_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,XGBoost,1,0.651163,0.907482,0.732359
1,XGBoost,2,0.619910,0.918350,0.737606
2,XGBoost,3,0.729032,0.917951,0.760119


In [33]:
def make_xgboost_deeper():
    return XGBClassifier(n_estimators=700, learning_rate=0.04, max_depth=8, min_child_weight=5, subsample=0.85, colsample_bytree=0.9, reg_lambda=2.0, reg_alpha=0.1, eval_metric="logloss", random_state=0, n_jobs=-1)

Во втором варианте разрешим более глубокие деревья и немного увеличим число итераций

In [34]:
xgboost_deeper_cv = evaluate_model_temporal(train_features, make_xgboost_deeper, "XGBoost deeper")
xgboost_deeper_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,XGBoost deeper,1,0.639269,0.905088,0.734038
1,XGBoost deeper,2,0.582278,0.920994,0.741568
2,XGBoost deeper,3,0.788732,0.921043,0.762287


XGBoost показал хорошее качество, однако не смог превзойти HistGradientBoosting в среднем по временным фолдам

## LightGBM

In [35]:
from lightgbm import LGBMClassifier

In [36]:
def make_lightgbm():
    return LGBMClassifier(n_estimators=600, learning_rate=0.04, num_leaves=31, max_depth=-1, min_child_samples=20, subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0, reg_alpha=0.1, random_state=0, n_jobs=-1, verbosity=-1)

В первом варианте ограничим число листьев и добавим регуляризацию. Также будем использовать только часть признаков и объектов при обучении, чтобы уменьшить риск переобучения

In [37]:
lightgbm_cv = evaluate_model_temporal(train_features, make_lightgbm, "LightGBM")
lightgbm_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,LightGBM,1,0.703518,0.905612,0.739290
1,LightGBM,2,0.695431,0.926595,0.757500
2,LightGBM,3,0.719745,0.922676,0.768036


In [39]:
def make_lightgbm_deeper():
    return LGBMClassifier(n_estimators=800, learning_rate=0.03, num_leaves=63, max_depth=-1, min_child_samples=15, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0, reg_alpha=0.1, random_state=0, n_jobs=-1, verbosity=-1)

Во втором варианте увеличим число листьев и итераций, чтобы модель могла учитывать более сложные взаимодействия между признаками

In [40]:
lightgbm_deeper_cv = evaluate_model_temporal(train_features, make_lightgbm_deeper, "LightGBM deeper")
lightgbm_deeper_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,LightGBM deeper,1,0.707071,0.903013,0.733603
1,LightGBM deeper,2,0.709845,0.920010,0.755883
2,LightGBM deeper,3,0.732026,0.926207,0.761104


LightGBM показал наиболее стабильный результат среди рассмотренных моделей. Более сложный вариант практически не проседает ни на одном временном фолде и немного превосходит HistGradientBoosting по среднему значению основной метрики

## CatBoost

In [41]:
from catboost import CatBoostClassifier

Сначала используем сравнительно неглубокие деревья, небольшой learning rate и L2-регуляризацию

In [42]:
def make_catboost():
    return CatBoostClassifier(iterations=600, learning_rate=0.05, depth=6, l2_leaf_reg=3.0, random_seed=0, verbose=False, thread_count=-1)

In [43]:
catboost_cv = evaluate_model_temporal(train_features, make_catboost, "CatBoost")
catboost_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,CatBoost,1,0.685990,0.922773,0.748892
1,CatBoost,2,0.691919,0.935206,0.770645
2,CatBoost,3,0.790210,0.931707,0.784234


Во втором варианте немного увеличим глубину деревьев и число итераций. Learning rate при этом уменьшим, чтобы более сложная модель обучалась постепенно и не слишком быстро подстраивалась под тренировочные данные

In [44]:
def make_catboost_deeper():
    return CatBoostClassifier(iterations=800, learning_rate=0.04, depth=8, l2_leaf_reg=5.0, random_seed=0, verbose=False, thread_count=-1)

In [45]:
catboost_deeper_cv = evaluate_model_temporal(train_features, make_catboost_deeper, "CatBoost deeper")
catboost_deeper_cv

,name,fold,precision_at_recall_70,roc_auc,pr_auc
0,CatBoost deeper,1,0.689320,0.918581,0.752662
1,CatBoost deeper,2,0.631818,0.933156,0.762539
2,CatBoost deeper,3,0.753333,0.926099,0.780675
